# 1. Bölüm: Teorik Temeller (Fonem Modelleme)

## Viterbi Algoritması Hesaplaması
**Gözlem Dizisi:** [High, Low]

### 1. Adım: t=1 (Gözlem: High)
* $V_{1}(e) = P(e) \times P(High|e) = 1.0 \times 0.7 = 0.7$
* $V_{1}(v) = P(v) \times P(High|v) = 0.0 \times 0.1 = 0$

### 2. Adım: t=2 (Gözlem: Low)
* **e durumuna gelme:** $V_{2}(e) = V_{1}(e) \times P(e \to e) \times P(Low|e) = 0.7 \times 0.6 \times 0.3 = 0.126$
* **v durumuna gelme:** $V_{2}(v) = V_{1}(e) \times P(e \to v) \times P(Low|v) = 0.7 \times 0.4 \times 0.9 = 0.252$

**Sonuç:** $V_{2}(v) > V_{2}(e)$ olduğu için en olası dizi **"e-v"** fonem dizisidir.

In [ ]:
from hmmlearn import hmm
import numpy as np

# 1. 'EV' KELİMESİ İÇİN MODEL [cite: 30-31]
# n_trials=1: Her adımda tek bir gözlem yapıldığını belirtir.
model_ev = hmm.MultinomialHMM(n_components=2, n_trials=1)
model_ev.startprob_ = np.array([1.0, 0.0]) # Başlangıç olasılığı P(e)=1.0 [cite: 18]

# Geçiş Olasılıkları (A Matrisi) [cite: 12-13]
model_ev.transmat_ = np.array([
    [0.6, 0.4], # e -> e, e -> v
    [0.2, 0.8]  # v -> e, v -> v
])

# Emisyon Olasılıkları (B Matrisi) [cite: 15-16]
model_ev.emissionprob_ = np.array([
    [0.7, 0.3], # e durumu: High(0.7), Low(0.3)
    [0.1, 0.9]  # v durumu: High(0.1), Low(0.9)
])

# 2. 'OKUL' KELİMESİ İÇİN MODEL (Karşılaştırma Amaçlı) [cite: 23, 34]
model_okul = hmm.MultinomialHMM(n_components=2, n_trials=1)
model_okul.startprob_ = np.array([1.0, 0.0])
model_okul.transmat_ = np.array([[0.5, 0.5], [0.5, 0.5]])
model_okul.emissionprob_ = np.array([[0.4, 0.6], [0.6, 0.4]])

# 3. TEST VERİSİ (Yeni bir ses kaydı: High, Low, Low) [cite: 17]
# MultinomialHMM formatı: [1, 0] -> High, [0, 1] -> Low
test_data = np.array([
    [1, 0], # High
    [0, 1], # Low
    [0, 1]  # Low
])

# 4. SKOR HESAPLAMA (Log-Likelihood) [cite: 26, 34]
score_ev = model_ev.score(test_data)
score_okul = model_okul.score(test_data)

# 5. SONUÇLARIN ANALİZİ VE YORUMLANMASI [cite: 35-37]
print("-" * 40)
print(f"EV Modeli Log-Likelihood Skoru: {score_ev}")
print(f"OKUL Modeli Log-Likelihood Skoru: {score_okul}")
print("-" * 40)

if score_ev > score_okul:
    print(f"KARAR: Ses verisi 'EV' kelimesi olarak tanındı. [cite: 26]")
    print(f"NEDEN: EV modelinin skoru ({score_ev:.4f}), OKUL modelinin skorundan ({score_okul:.4f}) ")
    print("daha yüksek (sıfıra daha yakın) olduğu için bu model veriyi daha iyi açıklar.")
else:
    print(f"KARAR: Ses verisi 'OKUL' kelimesi olarak tanındı. [cite: 26]")
    print(f"NEDEN: OKUL modelinin skoru ({score_okul:.4f}), EV modelinin skorundan ({score_ev:.4f}) ")
    print("daha yüksek olduğu için bu model veriyi daha iyi açıklar.")

MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


----------------------------------------
EV Modeli Log-Likelihood Skoru: -1.3295360273012822
OKUL Modeli Log-Likelihood Skoru: -2.3025850929940455
----------------------------------------
KARAR: Ses verisi 'EV' kelimesi olarak tanındı. [cite: 26]
NEDEN: EV modelinin skoru (-1.3295), OKUL modelinin skorundan (-2.3026) 
daha yüksek (sıfıra daha yakın) olduğu için bu model veriyi daha iyi açıklar.


3. Bölüm: Analiz ve Yorumlama

1. Ses verisindeki "gürültü" (noise), HMM modelindeki Emisyon Olasılıklarını nasıl etkiler?

Ses verisindeki gürültü, gözlemlenen sinyalin (High/Low) ayırt ediciliğini azaltır ve spektral özellikleri bozar.

Bu durum, HMM modelindeki Emisyon Olasılıkları matrisindeki değerlerin birbirine yaklaşmasına (olasılık farklarının azalmasına) yol açar.

Sonuç olarak, belirli bir fonem (durum) için doğru gözlemi üretme olasılığı düştüğü için modelin tanıma performansı azalır ve hata oranı artar.

2. Gerçek bir sistemde binlerce kelime olduğunu düşünürsek, Viterbi yerine neden daha karmaşık yapılar (Deep Learning gibi) tercih edilmeye başlanmıştır?

Kelime sayısı arttıkça, HMM modelleri fonemler arasındaki uzun vadeli bağımlılıkları ve bağlamsal bilgileri (contextual information) yakalamakta yetersiz kalır.

Viterbi algoritması, büyük kelime dağarcıklarında her bir kelime için ayrı ayrı olasılık yolu hesapladığından, arama uzayı büyüdükçe hesaplama maliyeti çok yükselir.

Derin Öğrenme (RNN, LSTM veya Transformer) modelleri ise devasa verilerden karmaşık öznitelikleri otomatik olarak öğrenebilir ve çok daha yüksek tanıma doğruluğu sunar.
